#non-risk 포함/제외가 TopN 결과에 "영향이 크지 않다"를 수치로 증명하기 위한 코드

In [ ]:
# 목적: non-risk 포함/제외가 TopN 결과에 "영향이 크지 않다"를 수치로 증명
# 1) non-risk 포함 TopN (actual_rate 기준)
# 2) non-risk 제외 TopN (conduct 건수 기준)
# 3) TopN 겹침률(Overlap) + 순위상관


import pandas as pd
import numpy as np
from scipy.stats import spearmanr


# 설정

CSV_PATH = "consumer_risk_analysis_regime_utf8.csv"
MIN_N = 100
TOPN = 30
GROUP_COLS = ["regime","sales_channel","contract_mode","age_group","item_group"]


# 0) 로드

df = pd.read_csv(CSV_PATH, encoding="utf-8")
df["y"] = (df["risk_type"].astype(str) == "conduct").astype(int)


# 1) (A) non-risk 포함 TopN
#    - 분모: 전체(=risk + non-risk)
#    - actual_rate = conduct 비율
combo_all = (
    df.groupby(GROUP_COLS, dropna=False)
      .agg(
          n=("y","size"),            # 전체 건수 (risk+non-risk)
          conduct=("y","sum"),       # conduct 건수
          actual_rate=("y","mean")   # conduct 비율
      )
      .reset_index()
)

combo_all_f = combo_all[combo_all["n"] >= MIN_N].copy()

top_all = (
    combo_all_f.sort_values(["actual_rate","n","conduct"], ascending=[False,False,False])
               .head(TOPN)
               .copy()
)
top_all["rank_all"] = np.arange(1, len(top_all) + 1)


# 2) (B) non-risk 제외 TopN
#    - risk_type == conduct만 남김
#    - "conduct 건수"로 TopN (분모 자체가 없으므로 비율 비교가 불가능)

df_risk = df[df["y"] == 1].copy()

combo_risk = (
    df_risk.groupby(GROUP_COLS, dropna=False)
           .size()
           .reset_index(name="conduct_only_n")  # conduct 건수
)

top_risk = (
    combo_risk.sort_values(["conduct_only_n"], ascending=[False])
              .head(TOPN)
              .copy()
)
top_risk["rank_risk"] = np.arange(1, len(top_risk) + 1)


# 3) TopN 겹침률(Overlap)

overlap = pd.merge(
    top_all[GROUP_COLS + ["rank_all", "actual_rate", "n", "conduct"]],
    top_risk[GROUP_COLS + ["rank_risk", "conduct_only_n"]],
    on=GROUP_COLS,
    how="inner"
)

overlap_cnt = len(overlap)
overlap_rate = overlap_cnt / TOPN

print("===== TopN 비교: non-risk 포함 vs 제외 =====")
print(f"- TopN 크기: {TOPN}")
print(f"- 겹치는 조합 수: {overlap_cnt}")
print(f"- 겹침 비율: {overlap_rate:.2%}")


# 4) 순위상관(겹치는 조합에 대해서만)

if overlap_cnt >= 3:
    rho, pval = spearmanr(overlap["rank_all"], overlap["rank_risk"])
    print(f"\n[겹치는 조합 내 순위 Spearman 상관] rho={rho:.3f}, p-value={pval:.3e}")
else:
    print("\n겹치는 조합이 너무 적어서 순위상관 계산 생략")


# 5) 전체 후보집합에서 관계 확인(상관)
#    - conduct_only_n(=conduct 건수)와 actual_rate의 관계(필터 통과 집합에서)

compare = pd.merge(
    combo_all_f,
    combo_risk,
    on=GROUP_COLS,
    how="left"
)

compare["conduct_only_n"] = compare["conduct_only_n"].fillna(0)

corr = compare[["actual_rate", "conduct", "conduct_only_n", "n"]].corr(numeric_only=True)
print("\n[상관행렬] (필터 n>=MIN_N 통과 집합)")
print(corr)

===== TopN 비교: non-risk 포함 vs 제외 =====
- TopN 크기: 30
- 겹치는 조합 수: 8
- 겹침 비율: 26.67%

[겹치는 조합 내 순위 Spearman 상관] rho=0.143, p-value=7.358e-01

[상관행렬] (필터 n>=MIN_N 통과 집합)
                actual_rate   conduct  conduct_only_n         n
actual_rate        1.000000  0.355683        0.355683  0.113258
conduct            0.355683  1.000000        1.000000  0.927143
conduct_only_n     0.355683  1.000000        1.000000  0.927143
n                  0.113258  0.927143        0.927143  1.000000

[해석 가이드]
- 겹침 비율이 높을수록(non-risk 포함/제외) TopN 구성은 크게 달라지지 않음
- 겹치는 조합에서 순위상관(rho)이 높을수록 '순서'도 유사함
- 상관행렬에서 actual_rate와 conduct_only_n(또는 conduct)가 함께 높게 움직이면
  non-risk(분모) 포함이 조합 위험 구조를 크게 왜곡하지 않는다는 근거로 사용 가능


In [ ]:
import pandas as pd
import numpy as np

CSV_PATH = "consumer_risk_analysis_regime_utf8.csv"
df = pd.read_csv(CSV_PATH)

df["y"] = (df["risk_type"] == "conduct").astype(int)

group_cols = ["regime","sales_channel","contract_mode","age_group","item_group"]

combo = (
    df.groupby(group_cols)
      .agg(
          n=("y","size"),
          conduct=("y","sum")
      )
      .reset_index()
)

combo["non_risk"] = combo["n"] - combo["conduct"]
combo["non_risk_ratio"] = combo["non_risk"] / combo["n"]

print("non-risk 비율 기술통계")
print(combo["non_risk_ratio"].describe())

print("\nnon-risk 비율 표준편차:", combo["non_risk_ratio"].std())

non-risk 비율 기술통계
count    4747.000000
mean        0.399949
std         0.309138
min         0.000000
25%         0.125000
50%         0.375000
75%         0.615385
max         1.000000
Name: non_risk_ratio, dtype: float64

non-risk 비율 표준편차: 0.30913752542980105


-----



# TopN_RF

In [ ]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.metrics import roc_auc_score
from sklearn.ensemble import RandomForestClassifier

df = pd.read_csv("/content/consumer_risk_analysis_regime_utf8.csv")

# 1) 타깃(y): conduct 여부
df["y"] = (df["risk_type"].astype(str) == "conduct").astype(int)

# 2) 사용할 변수(조합 축)
feat_cols = ["regime", "sales_channel", "contract_mode", "age_group", "item_group"]

# dropna 대신 NA 채우기(원핫 안정 + 데이터 손실 방지)
df_m = df.copy()
for c in feat_cols:
    df_m[c] = df_m[c].astype(str).fillna("NA").replace("nan", "NA")

X = df_m[feat_cols]
y = df_m["y"]

# 3) 전처리
preprocess = ColumnTransformer(
    transformers=[("cat", OneHotEncoder(handle_unknown="ignore", min_frequency=20), feat_cols)],
    remainder="drop"
)

# 4) 랜덤포레스트
rf = RandomForestClassifier(
    n_estimators=150,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced_subsample",
    max_depth=18,
    min_samples_leaf=20
)

pipe = Pipeline(steps=[("preprocess", preprocess), ("model", rf)])

# 5) 학습/검증 분리
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

pipe.fit(X_train, y_train)

# 6) 성능(AUC)
proba_test = pipe.predict_proba(X_test)[:, 1]
print("Test AUC:", roc_auc_score(y_test, proba_test))

# 7) 각 행 예측확률 부여
df_m["pred_risk"] = pipe.predict_proba(X)[:, 1]

# 8) 조합별 집계
combo = (
    df_m.groupby(feat_cols, dropna=False)
        .agg(
            n=("y", "size"),
            actual_rate=("y", "mean"),
            pred_risk=("pred_risk", "mean")
        )
        .reset_index()
)
min_n = 100
combo_f = combo[combo["n"] >= min_n].copy()

# 10) 전체 조합(필터 통과한 것 전부) 정렬
all_combo = combo_f.sort_values(["pred_risk", "n"], ascending=[False, False]).reset_index(drop=True)

# Define OUT_PATH before use
OUT_PATH = "topn_rf_predicted_risk.csv"

# 11) CSV 저장
all_combo.to_csv(OUT_PATH, index=False, encoding="utf-8-sig")
print(f"CSV 저장 완료: {OUT_PATH}")

# 12) 화면 출력
all_combo